# 02 - DKT Next Item

Objetivos:
- reproduzir o treino do DKT next-item
- validar construção do dataset
- acompanhar métricas de validação e teste
- comparar com versões futuras (LPKT)


In [ ]:
from __future__ import annotations

import json
import random
import sys
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from brain_kt.dataset.kt_next_item_dataset import KTNextItemDataset, build_id_mappings
from brain_kt.models.dkt_next_item import DKTNextItemModel
from brain_kt.preprocessing.build_next_item_training_sequences import build_next_item_training_sequences

INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'sequences' / 'user_sequences.json'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

In [ ]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def compute_auc(probs: torch.Tensor, targets: torch.Tensor) -> float:
    probs = probs.detach().cpu()
    targets = targets.detach().cpu()

    pos = probs[targets == 1]
    neg = probs[targets == 0]

    if len(pos) == 0 or len(neg) == 0:
        return 0.5

    correct = 0.0
    total = 0.0

    for p in pos:
        correct += (p > neg).sum().item()
        total += len(neg)

    return correct / total


def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> tuple[float, float]:
    model.eval()
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(batch)
            probs = torch.sigmoid(logits)
            mask = batch['mask']
            all_probs.append(probs[mask])
            all_targets.append(batch['targets'][mask])

    probs = torch.cat(all_probs)
    targets = torch.cat(all_targets)
    auc = compute_auc(probs, targets)
    acc = ((probs > 0.5) == targets).float().mean().item()
    return auc, acc

## Carregamento e construção do dataset

In [ ]:
set_seed(42)

with open(INPUT_PATH, 'r', encoding='utf-8') as f:
    user_sequences = json.load(f)

training_sequences = build_next_item_training_sequences(
    user_sequences,
    max_seq_len=100,
    stride=50,
)

len(user_sequences), len(training_sequences)

In [ ]:
random.shuffle(training_sequences)
n = len(training_sequences)

train_data = training_sequences[: int(0.7 * n)]
val_data = training_sequences[int(0.7 * n): int(0.85 * n)]
test_data = training_sequences[int(0.85 * n):]

print('train:', len(train_data))
print('val:', len(val_data))
print('test:', len(test_data))

In [ ]:
mappings = build_id_mappings(train_data)

train_ds = KTNextItemDataset(train_data, mappings.question_to_idx, mappings.skill_to_idx)
val_ds = KTNextItemDataset(val_data, mappings.question_to_idx, mappings.skill_to_idx)
test_ds = KTNextItemDataset(test_data, mappings.question_to_idx, mappings.skill_to_idx)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

len(train_ds), len(val_ds), len(test_ds)

In [ ]:
sample = train_ds[0]
{k: tuple(v.shape) for k, v in sample.items()}

## Treino do modelo

In [ ]:
model = DKTNextItemModel(
    num_questions=len(mappings.question_to_idx),
    num_skills=len(mappings.skill_to_idx),
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss(reduction='none')

history = []
best_val_auc = 0.0
best_state = None

In [ ]:
for epoch in range(10):
    model.train()

    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(batch)
        loss = criterion(logits, batch['targets'])
        loss = (loss * batch['mask']).sum() / batch['mask'].sum()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    val_auc, val_acc = evaluate(model, val_loader, DEVICE)
    history.append({'epoch': epoch + 1, 'val_auc': val_auc, 'val_acc': val_acc})

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(f"Epoch {epoch+1} | Val AUC: {val_auc:.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
history_df = pd.DataFrame(history)
history_df

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history_df['epoch'], history_df['val_auc'], marker='o')
plt.title('DKT Next Item - Val AUC por época')
plt.xlabel('Época')
plt.ylabel('Val AUC')
plt.show()

## Avaliação final

In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)

test_auc, test_acc = evaluate(model, test_loader, DEVICE)

print(f'Test AUC: {test_auc:.4f}')
print(f'Test Acc: {test_acc:.4f}')

## Conclusões

Preencher após execução:
- melhor Val AUC
- Test AUC
- comparação com LPKT next-item
- implicações para recomendação adaptativa
